In [15]:
from datasets import load_dataset

# load reddits explain like am 5 dataset
eli5 = load_dataset("dany0407/eli5_category", split="train[:5000]")

In [16]:
# dataset strcutrue
eli5

Dataset({
    features: ['q_id', 'title', 'selftext', 'category', 'subreddit', 'answers', 'title_urls', 'selftext_urls'],
    num_rows: 5000
})

In [17]:
eli5[0]

{'q_id': '5lchat',
 'title': "Why there was a 'leap second' added to the end of 2016?",
 'selftext': '',
 'category': 'Other',
 'subreddit': 'explainlikeimfive',
 'answers': {'a_id': ['dbuoyxl', 'dbur7gi', 'dbuotht'],
  'text': ['the rotation of the earth is not a constant. in fact the rotation of the earth is slowing down, which means that a full day is getting slightly longer. without leap seconds our clocks would slowly drift ever so slightly out of sync with the actual day. we could deal with this by redefining how how long 1 second is, making it slightly longer so that one day is still exactly 24*60*60 seconds. but in practice that is really inconvenient for a lot of our technology which relies on very precise timing. its easier to just move us ahead one second every couple of years or so.',
   "The Earth's rotation is not regular. It varies a bit, so sometimes we add a second. We do this to ensure that noon is always going to be sometime around mid-day. If we did not add leap sec

### Split the dataset’s `train` split into a `train` and `test` set with the `train_test_split` method
**Meaning, You are creating a separate "test" slice of data that the model won't see during its initial training.**
  
Think of it like preparing for a final exam. If you study using the exact same questions that are on the test, you haven't actually learned the material - you’ve just memorized the answers. In machine learning, we call that `overfitting`.

**Why do this?**

We need two distinct "folders" for our data to ensure the AI is actually getting smarter:

- **Training Set (The Textbook):** This is what the model "reads" to learn patterns, grammar, and facts.

- **Test Set (The Exam):** This is a set of data the model never sees during training. After the model is finished learning, we show it these test questions. If it answers them correctly, we know it has actually "understood" how to explain things simply, rather than just memorizing the training data.

In [18]:
eli5 = eli5.train_test_split(test_size=0.2) # put 20% into test split

In [19]:
eli5

DatasetDict({
    train: Dataset({
        features: ['q_id', 'title', 'selftext', 'category', 'subreddit', 'answers', 'title_urls', 'selftext_urls'],
        num_rows: 4000
    })
    test: Dataset({
        features: ['q_id', 'title', 'selftext', 'category', 'subreddit', 'answers', 'title_urls', 'selftext_urls'],
        num_rows: 1000
    })
})

### Load DistilGPT2 tokenizer to process the `text` subfiled

In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2") 

In [21]:
eli5 = eli5.flatten()

eli5["train"][0] # notice the answer.text, its flattened

{'q_id': '79ub8l',
 'title': 'why is gold backed currency dangerous to global financial order?',
 'selftext': '',
 'category': 'Economics',
 'subreddit': 'explainlikeimfive',
 'answers.a_id': ['dp4wgfp'],
 'answers.text': ['A gold backed currency isn\'t dangerous on its own. The US used to have a gold backed system until the great depression. Back then exchange rates were fixed bc almost everybody backed their currency with gold. Some conservatives like this idea because it prevents infinite devaluation of a currency: There is a finite amount of gold in the world, certain countries own certain amounts so they can only make as much money as gold they own. The main problem with gaddafi trying to go back to this system is that oil is traded in US dollars. Besides the US liking this way of business, countries now only use gold as a form of collateral between them as they make deals and their currencies "float" ever since the 70\'s which means they are traded on exchanges in the global fina

In [22]:
# join all the answers per post, using a preprocess function
def preprocess_function(examples: [str]):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])

In [23]:
# apply this preprocessing function over the entire dataset
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5["train"].column_names
)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1069 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1334 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (4615 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1129 > 1024). Running this sequence through the model will result in indexing errors


Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1758 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2167 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1060 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1282 > 1024). Running this sequence through the model will result in indexing errors


In [24]:
len(tokenized_eli5["train"][0]["input_ids"])

289

In [25]:
# examples = tokenized_eli5.map()

In [26]:
# examples

In [27]:
# examples.column_names

This dataset contains the token sequences, but some of these are longer than the maximum input length for the model.

You can now use a second preprocessing function to

- concatenate all the sequences
- split the concatenated sequences into shorter chunks defined by block_size, which should be both shorter than the maximum input length and short enough for your GPU RAM.

In [28]:
block_size = 128


def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [29]:
# Apply the group_texts function over the entire dataset:
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [30]:
# Batching. We using end-of-sequence tokens as padding tokens
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False) # disable masking

In [31]:
# Training
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [34]:
# training arguments

training_args = TrainingArguments(
    output_dir="../models/causal_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=4
)

# calling the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [35]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.910924,3.820010
2,3.825938,3.811035
3,3.784156,3.809557


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3906, training_loss=3.8430663312451996, metrics={'train_runtime': 637.9967, 'train_samples_per_second': 48.955, 'train_steps_per_second': 6.122, 'total_flos': 1020135176404992.0, 'train_loss': 3.8430663312451996, 'epoch': 3.0})

In [37]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

RuntimeError: on_train_begin must be called before on_evaluate

In [48]:
prompt = "How does Spotify and Apple Music pay artists?"

In [51]:
from transformers import pipeline

generator = pipeline("text-generation", model="../models/causal_model/checkpoint-3906")
generator(prompt)[0]["generated_text"]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"How does Spotify and Apple Music pay artists? They give them a nice, free pass to show songs to their music providers. If you do the same thing, you get a free pass to show them a song later. So Spotify pays artists a ton more. You need a decent pass to show their songs to their music providers. If you do that, you get a free pass to show them a song later. (In other words, if you do that, you get a free pass to show them a song later.) Spotify pays artists as much as they get paid. It's like paying the artist a ton more for having a song later. This is similar to paying an artist a ton of money for a song later (like paying for a song later). Spotify pays artists for access to song metadata, music, and content, but it doesn't pay artists for access to song metadata. The reason Spotify pays artists for access to song metadata is because they are paid for it. Now, they don't pay artists for access to song metadata, so that's not really a reason for Spotify to pay artists for access to 